# Study 879 — Weekly Economic Index — the teardown

The predictive-regression Newey-West (8-lag) HAC *t*, the two-era cut, the 2,000-draw permutation placebo, the costed rotation overlay, and the 20-seed synthetic control.

In [1]:
R = {'start': '2008-01-12', 'end': '2026-06-13', 'n': 962, 'fingerprint': '94e9e76c22ef', 'spy1_lvl_t': -1.12, 'spy1_lvl_uni': -0.96, 'spy1_dwei_t': 1.34, 'spy1_dwei_uni': 1.16, 'spy1_r2': 0.0033, 'spy4_lvl_t': -1.76, 'spy4_dwei_t': 0.69, 'spy4_r2': 0.0109, 'rot1_lvl_t': -1.88, 'rot1_dwei_t': 0.75, 'rot1_r2': 0.005, 'rot4_lvl_t': -2.24, 'rot4_lvl_uni': -2.22, 'rot4_dwei_t': 0.06, 'rot4_r2': 0.0202, 'era_spy1_early_lvl': 0.45, 'era_spy1_early_dwei': 2.26, 'era_spy1_late_lvl': -2.69, 'era_spy1_late_dwei': 0.24, 'era_rot4_early_lvl': -0.59, 'era_rot4_late_lvl': -2.52, 'cond_spy1_dwei': 0.4, 'cond_spy1_base': 0.24, 'cond_spy1_welch': 1.22, 'cond_rot4_wei': 0.05, 'cond_rot4_base': 0.36, 'cond_rot4_welch': -1.16, 'placebo_obs_t': 2.22, 'placebo_mean_t': 0.8, 'placebo_p': 0.026, 'ov_wei_gross': -0.11, 'ov_wei_net': -0.16, 'ov_dwei_gross': 0.21, 'ov_dwei_net': 0.02, 'ov_hold': 0.28, 'ov_dwei_turns': 1123, 'null_mean_t': -0.13, 'null_sd_t': 1.12, 'null_fire': 1, 'planted_spy_t': 16.69, 'planted_rot_t': 22.05}

## The headline — predictive regression (Newey-West HAC *t*)

Forward return on a constant + standardized WEI level & weekly change.

In [2]:
print(f"SPY 1wk    : R2={R['spy1_r2']:+.4f}  level t={R['spy1_lvl_t']:+.2f}  dwei t={R['spy1_dwei_t']:+.2f}")
print(f"SPY 4wk    : R2={R['spy4_r2']:+.4f}  level t={R['spy4_lvl_t']:+.2f}  dwei t={R['spy4_dwei_t']:+.2f}")
print(f"XLY-XLP 1wk: R2={R['rot1_r2']:+.4f}  level t={R['rot1_lvl_t']:+.2f}  dwei t={R['rot1_dwei_t']:+.2f}")
print(f"XLY-XLP 4wk: R2={R['rot4_r2']:+.4f}  level t={R['rot4_lvl_t']:+.2f}  dwei t={R['rot4_dwei_t']:+.2f}  <- only |t|>=2, WRONG sign")

SPY 1wk    : R2=+0.0033  level t=-1.12  dwei t=+1.34
SPY 4wk    : R2=+0.0109  level t=-1.76  dwei t=+0.69
XLY-XLP 1wk: R2=+0.0050  level t=-1.88  dwei t=+0.75
XLY-XLP 4wk: R2=+0.0202  level t=-2.24  dwei t=+0.06  <- only |t|>=2, WRONG sign


## Robustness — two eras (split 2017-01-01), univariate HAC *t*

The one claim-consistent hit (weekly change -> SPY) is an early-era artefact.

In [3]:
print(f"SPY 1wk level: early {R['era_spy1_early_lvl']:+.2f}  ->  late {R['era_spy1_late_lvl']:+.2f}")
print(f"SPY 1wk dwei : early {R['era_spy1_early_dwei']:+.2f}  ->  late {R['era_spy1_late_dwei']:+.2f}   (dies after 2017)")
print(f"rot 4wk level: early {R['era_rot4_early_lvl']:+.2f}  ->  late {R['era_rot4_late_lvl']:+.2f}   (wrong sign, late-driven)")

SPY 1wk level: early +0.45  ->  late -2.69
SPY 1wk dwei : early +2.26  ->  late +0.24   (dies after 2017)
rot 4wk level: early -0.59  ->  late -2.52   (wrong sign, late-driven)


## Placebo — permute the nowcast, re-run the HAC slope *t* (rot_h4 level, 2,000 draws)

In [4]:
print(f"observed |t| = {R['placebo_obs_t']:.2f} vs placebo mean |t| = {R['placebo_mean_t']:.2f} -> two-sided p = {R['placebo_p']:.4f}")
print('  the wrong-signed rotation relation is real (mean-reversion) but OPPOSITE to the claim')

observed |t| = 2.22 vs placebo mean |t| = 0.80 -> two-sided p = 0.0260
  the wrong-signed rotation relation is real (mean-reversion) but OPPOSITE to the claim


## The timer — costed long-cyclical / short-defensive overlay, weekly

One-way 5 bps/leg on turnover + 50 bps/yr borrow on the short; raced vs always-hold.

In [5]:
print(f"WEI>median : gross Sh {R['ov_wei_gross']:+.2f}  net Sh {R['ov_wei_net']:+.2f}  vs hold {R['ov_hold']:+.2f}")
print(f"dwei>0     : gross Sh {R['ov_dwei_gross']:+.2f}  net Sh {R['ov_dwei_net']:+.2f}  vs hold {R['ov_hold']:+.2f}  ({R['ov_dwei_turns']} turns eat it)")

WEI>median : gross Sh -0.11  net Sh -0.16  vs hold +0.28
dwei>0     : gross Sh +0.21  net Sh +0.02  vs hold +0.28  (1123 turns eat it)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null (beyond ~5%) and must recover a planted relation.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from wei import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic(edge=0.0, seed=879+s, n=700))['t_level'] for s in range(10)])
print(f"null (edge=0), 10 seeds: level t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/10")
planted = st.synthetic_detect(data.synthetic(edge=0.010, seed=879, n=700))
print(f"planted (edge=0.010): SPY level t = {planted['t_level']:+.2f}  (recovers cleanly)")

null (edge=0), 10 seeds: level t mean -0.03 (sd 1.40), |t|>=2 in 1/10
planted (edge=0.010): SPY level t = +16.69  (recovers cleanly)


## Verdict

- **Signal — None.** A weekly growth nowcast does **not** beat the monthly tape. The WEI level is insignificant/wrong-signed on forward SPY (*t* = -1.12) and era-unstable; the weekly change is the right sign but insignificant (*t* = +1.34), its only \|t\| ≥ 2 hit a 2008–09-recovery artefact that dies post-2017 (+2.26 → +0.24). The single overall \|t\| ≥ 2 slope — rotation level, -2.24 (placebo p = 0.026) — is *wrong-signed* (a mean-reversion). The 20-seed synthetic control recovers a planted edge (*t* = +16.69) and fires on the null at ~5% (1/20), so the null is genuine.
- **Tradability — Mirage.** The costed XLY−XLP overlay under-performs always-hold (net Sharpe -0.16 / +0.02 vs +0.28); the thin weekly-change edge is eaten by 1123 weekly turns.